# ACC 380K — Your first decision tree

In class we grew a decision tree by hand from eight students. This notebook grows the same tree with scikit-learn, draws it, and uses it to predict Rey.

**Notebook version 0.1.1 · November 12, 2026**

Run the cells from top to bottom. In Colab, choose **File → Save a copy in Drive** if you want your own copy. There is no API key and no file to upload. A full run takes a few seconds.

## What you should be able to do

- Load a small table and turn Yes/No answers into True/False.
- Fit a `DecisionTreeClassifier` and read the picture it draws.
- Predict a new student and trace the path that gives the answer.
- Say why a perfect score on the training students is weak evidence.

## Setup

`DecisionTreeClassifier` grows the tree. `plot_tree` draws it.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.tree import DecisionTreeClassifier, plot_tree

DATA_URL = "https://acc380k-data.nicholashallman.workers.dev/public/rf/silly_data.csv"

## Load the eight students

Each row is one student. The last column, `a_grade`, is what we want to predict.

In [ ]:
# The data server turns away requests that do not say who is asking, so we give a name.
students = pd.read_csv(DATA_URL, storage_options={"User-Agent": "acc380k-notebook"})
students

## Turn Yes/No into True/False

scikit-learn needs numbers, not text. Python treats `True` as 1 and `False` as 0, so True/False columns work.

In [ ]:
for column in ["perfect_attendance", "prior_coding", "a_grade"]:
    students[column] = students[column].map({"No": False, "Yes": True})

students

## Fit the tree

`X` holds the three facts the tree may ask about. `y` holds the answer. `fit` tries every question, scores each one with Gini impurity, keeps the best, and repeats inside each group. That is what we did on the slides.

In [ ]:
FEATURES = ["perfect_attendance", "hours_per_week", "prior_coding"]

X = students[FEATURES]
y = students["a_grade"]

tree = DecisionTreeClassifier()
tree.fit(X, y)
print("Questions deep:", tree.get_depth(), "| leaves:", tree.get_n_leaves())

## Draw the tree

How to read each box:

- The first line is the question. `prior_coding <= 0.5` means "no prior coding", because False is 0 and True is 1.
- If the answer is yes, follow the arrow marked True, to the **left**. If no, go right.
- `samples` is how many students are in the group.
- `gini` is the mixture in that group. 0 means everyone in it got the same grade.
- `value` splits that count into [no A, A], and `class` is the majority.

It is the tree from the slides, drawn as a mirror image: scikit-learn asks "no prior coding?" where we asked "prior coding?".

In [ ]:
plt.figure(figsize=(9, 6))
plot_tree(tree, feature_names=FEATURES, class_names=["no A", "A"], rounded=True, fontsize=11)
plt.show()

## Predict new students

Rey has perfect attendance, studies 3 hours a week, and has no prior coding. Finn is the same, but studies 5 hours. Trace each one down the picture before you run the cell.

In [ ]:
new_students = pd.DataFrame(
    {"perfect_attendance": [True, True], "hours_per_week": [3, 5], "prior_coding": [False, False]},
    index=["Rey", "Finn"],
)

new_students["predicted_a_grade"] = tree.predict(new_students[FEATURES])
new_students

## Check the tree on the students it learned from

`score` gives the share of students the tree gets right.

In [ ]:
print("Right on the training students:", f"{tree.score(X, y):.0%}")

## Read the result

The tree gets all eight students right. That sounds better than it is. The tree kept asking questions until every group was pure, so it had to get these eight right. It has not been tested on anyone new.

Rey's prediction rests on the three students in her leaf. With a few different students, the tree could look completely different. The first exercise below shows it. Random forests exist to fix this.

## Try it

1. **Drop two students.** In the fit cell, change the two lines to `X = students[FEATURES].head(6)` and `y = students["a_grade"].head(6)`. Rerun the fit and draw cells. Which question disappears? What does the tree now predict for Finn?
2. **Limit the tree.** Put the two lines back, then use `DecisionTreeClassifier(max_depth=1)`. Which question survives, and what is the training score now?
3. **Change Rey.** In the predict cell, give Rey prior coding. Does her prediction change? Does the number of hours still matter?